# RL Experiment 09: Multi-Agent Systems

**Self-contained experiment notebook using DRY architecture.**

## Architecture (DRY Principle)

| Location | What | Example |
|----------|------|---------|
| `schedule_engine/notebooks/` | Reusable functions | `load_context()`, `create_env()` |
| `src/schedule_engine/rl/multi_agent/` | Multi-agent coordinator | `AgentCoordinator` |
| **This notebook** | Experiment-specific config | Multi-episode selection analysis |

## Experiment Overview
- **System**: Agent coordinator with state-based specialist selection
- **Goal**: Analyze agent selection dynamics across multiple episodes
- **Metrics**: Selection distribution, state transitions, diversity tracking

## 1. Imports (from `schedule_engine/notebooks/`)

In [ ]:
from __future__ import annotations
from collections import Counter
from datetime import datetime
from pathlib import Path

# DRY IMPORTS FROM schedule_engine/notebooks/
from schedule_engine.notebooks import (
    build_notebook_config,
    create_env,
    load_context,
    set_global_seed,
)
from schedule_engine.rl.multi_agent.agent_coordinator import AgentCoordinator

print(" All imports from schedule_engine/notebooks/ successful!")

## 2. Configuration (Inline - Experiment-Specific)

In [ ]:

# RL EXPERIMENT 09 CONFIGURATION - Multi-Agent Systems


SEED = 42
POP_SIZE = 20
MAX_GENERATIONS = 50
MAX_STEPS = 15
NUM_EPISODES = 10  # Episodes to run for analysis

# Paths - Organized by experiment with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/notebooks/rl_09_multi_agent_{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Config: pop={POP_SIZE}, ngen={MAX_GENERATIONS}, episodes={NUM_EPISODES}")
print(f" Output: {OUTPUT_DIR}")

## 3. Load Data & Create Environment

In [ ]:
# Set reproducibility
set_global_seed(SEED)

# Build config and load scheduling context
config = build_notebook_config(seed=SEED, overrides={"pop_size": POP_SIZE})
_, context = load_context(DATA_DIR, config)

# Create RL environment
env = create_env(
    context=context,
    pop_size=POP_SIZE,
    max_generations=MAX_GENERATIONS,
    max_steps=MAX_STEPS,
)

print(f" Environment created: obs_space={env.observation_space.shape}, action_space={env.action_space.n}")

## 4. Run Multi-Episode Selection Analysis

In [ ]:
# Create coordinator with state-based selection
coordinator = AgentCoordinator(strategy="state_based")

# Track selections and state transitions
all_selections = []
episode_data = []

for episode in range(NUM_EPISODES):
    obs, info = env.reset()
    episode_selections = []
    episode_states = []
    
    for step in range(MAX_STEPS):
        # Get current state for agent selection
        state = {
            "generation": info.get("generation", 0),
            "generations_without_improvement": info.get("generations_without_improvement", 0),
        }
        
        # Select specialist agent
        agent = coordinator.select_agent(env.population, state, obs)
        episode_selections.append(agent.name)
        episode_states.append(state.copy())
        
        # Take random action (for demonstration)
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        
        if terminated or truncated:
            break
    
    all_selections.extend(episode_selections)
    episode_data.append({
        "episode": episode,
        "selections": episode_selections,
        "states": episode_states,
        "num_steps": len(episode_selections),
    })
    
    print(f"Episode {episode+1}/{NUM_EPISODES}: {len(episode_selections)} steps, agents={episode_selections[:3]}...")

print(f"\n Completed {NUM_EPISODES} episodes, {len(all_selections)} total selections")

## 5. Results Summary

In [ ]:
# Analyze selection distribution
selection_counts = Counter(all_selections)

print(f"\n{'='*60}")
print(f"RL EXPERIMENT 09: MULTI-AGENT SYSTEMS RESULTS")
print(f"{'='*60}")
print(f"\nTotal selections: {len(all_selections)}")
print(f"Episodes: {NUM_EPISODES}")
print(f"\nAgent Selection Distribution:")
for agent_name, count in selection_counts.most_common():
    pct = 100 * count / len(all_selections)
    print(f"  {agent_name:20s}: {count:4d} ({pct:5.1f}%)")

# Episode statistics
steps_per_episode = [ed["num_steps"] for ed in episode_data]
print(f"\nSteps per Episode:")
print(f"  Mean: {sum(steps_per_episode)/len(steps_per_episode):.1f}")
print(f"  Min:  {min(steps_per_episode)}")
print(f"  Max:  {max(steps_per_episode)}")
print(f"{'='*60}")

## 6. Save Results

In [ ]:
import json

# Save experiment results
results_data = {
    "experiment": "rl_09_multi_agent_systems",
    "timestamp": TIMESTAMP,
    "config": {
        "seed": SEED,
        "pop_size": POP_SIZE,
        "max_generations": MAX_GENERATIONS,
        "max_steps": MAX_STEPS,
        "num_episodes": NUM_EPISODES,
        "strategy": "state_based",
    },
    "results": {
        "total_selections": len(all_selections),
        "selection_distribution": dict(selection_counts),
        "steps_per_episode": steps_per_episode,
        "episode_selections": [ed["selections"] for ed in episode_data],
    },
}

results_path = OUTPUT_DIR / "results.json"
with open(results_path, "w") as f:
    json.dump(results_data, f, indent=2)

print(f" Results saved to: {results_path}")